## Introduction

Data preprations and combining from 2017 to 2020.

## Data Prepration

In [1]:
import os
import shutil
import pandas as pd
from tqdm import tqdm
from skin_lesion_project.utils import eda_utils

In [2]:
class_labels = eda_utils.LABEL_COLS 
class_names = eda_utils.CLASS_NAMES

class_labels, class_names

(['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC'],
 {'MEL': 'Melanoma',
  'NV': 'Melanocytic Nevi',
  'BCC': 'Basal Cell Carcinoma',
  'AK': 'Actinic Keratosis',
  'BKL': 'Benign Keratosis',
  'DF': 'Dermatofibroma',
  'VASC': 'Vascular Lesion',
  'SCC': 'Squamous Cell Carcinoma'})

## Paths

In [3]:
data_2017 = pd.read_csv("/mnt/d/skin-lesion-data/2017/ISIC-2017_Training_Part3_GroundTruth.csv")
data_2018 = pd.read_csv("/mnt/d/skin-lesion-data/2018/ISIC2018_Task3_Training_GroundTruth/ISIC2018_Task3_Training_GroundTruth/ISIC2018_Task3_Training_GroundTruth.csv")

In [4]:
images_2017 = os.listdir("/mnt/d/skin-lesion-data/2017/ISIC-2017_Training_Data/ISIC-2017_Training_Data/")

## Renaming

In [5]:
# for img in tqdm(images_2017):
#     dst_name = img.split(".")[0]+"_2017.jpg"
#     os.rename(src=f"/mnt/d/skin-lesion-data/2017/ISIC-2017_Training_Data/ISIC-2017_Training_Data/{img}",
#               dst=f"/mnt/d/skin-lesion-data/2017/ISIC-2017_Training_Data/ISIC-2017_Training_Data/{dst_name}")

In [6]:
images_2018 = os.listdir("/mnt/d/skin-lesion-data/2018/ISIC2018_Task3_Training_Input/ISIC2018_Task3_Training_Input/")

In [7]:
# for img in tqdm(images_2018):
#     dst_name = img.split(".")[0]+"_2018.jpg"
#     os.rename(src=f"/mnt/d/skin-lesion-data/2018/ISIC2018_Task3_Training_Input/ISIC2018_Task3_Training_Input/{img}",
#               dst=f"/mnt/d/skin-lesion-data/2018/ISIC2018_Task3_Training_Input/ISIC2018_Task3_Training_Input/{dst_name}")

In [8]:
data_2017['image_id'] = data_2017['image_id'].apply(lambda x: x+"_2017.jpg")

In [9]:
data_2017

,image_id,melanoma,seborrheic_keratosis
0,ISIC_0000000_2017.jpg,0.0,0.0
1,ISIC_0000001_2017.jpg,0.0,0.0
2,ISIC_0000002_2017.jpg,1.0,0.0
3,ISIC_0000003_2017.jpg,0.0,0.0
4,ISIC_0000004_2017.jpg,1.0,0.0
...,...,...,...
1995,ISIC_0015220_2017.jpg,0.0,1.0
1996,ISIC_0015233_2017.jpg,0.0,1.0
1997,ISIC_0015260_2017.jpg,0.0,1.0
1998,ISIC_0015284_2017.jpg,1.0,0.0


In [10]:
data_2018['image'] = data_2018['image'].apply(lambda x: x+"_2018.jpg")

In [11]:
data_2018

,image,MEL,NV,BCC,AKIEC,BKL,DF,VASC
0,ISIC_0024306_2018.jpg,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,ISIC_0024307_2018.jpg,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,ISIC_0024308_2018.jpg,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,ISIC_0024309_2018.jpg,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,ISIC_0024310_2018.jpg,1.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...
10010,ISIC_0034316_2018.jpg,1.0,0.0,0.0,0.0,0.0,0.0,0.0
10011,ISIC_0034317_2018.jpg,1.0,0.0,0.0,0.0,0.0,0.0,0.0
10012,ISIC_0034318_2018.jpg,0.0,0.0,0.0,0.0,1.0,0.0,0.0
10013,ISIC_0034319_2018.jpg,0.0,1.0,0.0,0.0,0.0,0.0,0.0


## Label Flattening

In [12]:
labels_2018_ = {im: class_labels[int(data_2018.loc[data_2018["image"] == im].drop("image", axis=1).values[0].argmax())] for im in data_2018["image"].values}

In [13]:
mapping = {"melanoma": "MEL", "seborrheic_keratosis": "BKL"}

In [14]:
data_2017 = data_2017.rename({
    "melanoma": "MEL",
    "seborrheic_keratosis": "BKL",
}, axis=1)

In [15]:
data_2017 = data_2017[~(data_2017['MEL']==0) & (data_2017['BKL']==0)]

In [16]:
labels_2017_ = {im: class_labels[int(data_2017.loc[data_2017["image_id"] == im].drop("image_id", axis=1).values[0].argmax())] for im in data_2017["image_id"].values}

In [17]:
labels = pd.concat([pd.DataFrame({'image':labels_2017_.keys(), 'label':labels_2017_.values()}), pd.DataFrame({'image':labels_2018_.keys(), 'label':labels_2018_.values()})], axis=0)

In [18]:
labels

,image,label
0,ISIC_0000002_2017.jpg,MEL
1,ISIC_0000004_2017.jpg,MEL
2,ISIC_0000013_2017.jpg,MEL
3,ISIC_0000022_2017.jpg,MEL
4,ISIC_0000026_2017.jpg,MEL
...,...,...
10010,ISIC_0034316_2018.jpg,MEL
10011,ISIC_0034317_2018.jpg,MEL
10012,ISIC_0034318_2018.jpg,BKL
10013,ISIC_0034319_2018.jpg,NV


## Combining Final Data

In [34]:
# for img in tqdm(images_2017):
#     shutil.copy(src=f"/mnt/d/skin-lesion-data/2017/ISIC-2017_Training_Data/ISIC-2017_Training_Data/{img}",
#                 dst=f"/mnt/d/skin-lesion-data/final-data/train/{img}")

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2000/2000 [07:18<00:00,  4.56it/s]


In [37]:
# for img in tqdm(images_2018):
#     shutil.copy(src=f"/mnt/d/skin-lesion-data/2018/ISIC2018_Task3_Training_Input/ISIC2018_Task3_Training_Input/{img}",
#                 dst=f"/mnt/d/skin-lesion-data/final-data/train/{img}")

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10017/10017 [11:12<00:00, 14.90it/s]


In [4]:
# for im in tqdm(os.listdir("/mnt/d/skin-lesion-data/2019+2020/train/")):
#     shutil.copy(src=f"/mnt/d/skin-lesion-data/2019+2020/train/{im}",
#                 dst=f"/mnt/d/skin-lesion-data/final-data/train/{im}")

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 31332/31332 [42:34<00:00, 12.27it/s]


In [19]:
labels.to_csv("/mnt/d/skin-lesion-data/final-data/labels/labels-2017-2018.csv")

In [2]:
import hashlib

IMAGE_DIR = "/mnt/d/skin-lesion-data/final-data/train/"

hash_map = {}
duplicates = []

for filename in tqdm(os.listdir(IMAGE_DIR)):
    path = os.path.join(IMAGE_DIR, filename)

    if not os.path.isfile(path):
        continue

    with open(path, "rb") as f:
        file_hash = hashlib.md5(f.read()).hexdigest()

    if file_hash in hash_map:
        duplicates.append({
            "original": hash_map[file_hash],
            "duplicate": filename
        })
    else:
        hash_map[file_hash] = filename

duplicates_df = pd.DataFrame(duplicates)

print(f"Found {len(duplicates_df)} duplicate images.")

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 43346/43346 [55:13<00:00, 13.08it/s]

Found 10789 duplicate images.


In [3]:
duplicates_df

,original,duplicate
0,ISIC_0000000_2017.jpg,ISIC_0000000_2019.jpg
1,ISIC_0000001_2017.jpg,ISIC_0000001_2019.jpg
2,ISIC_0000002_2017.jpg,ISIC_0000002_2019.jpg
3,ISIC_0000003_2017.jpg,ISIC_0000003_2019.jpg
4,ISIC_0000004_2017.jpg,ISIC_0000004_2019.jpg
...,...,...
10784,ISIC_6352565_2020.jpg,ISIC_6584579_2020.jpg
10785,ISIC_6689310_2020.jpg,ISIC_7528783_2020.jpg
10786,ISIC_0149568_2020.jpg,ISIC_9022005_2020.jpg
10787,ISIC_9432244_2020.jpg,ISIC_9645937_2020.jpg


In [7]:
for im in tqdm(duplicates_df['duplicate'].values):
    os.remove(f"/mnt/d/skin-lesion-data/final-data/train/{im}")

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10789/10789 [00:48<00:00, 223.49it/s]
